# THE DISCOMBOBULATOR API SERVER
## by Rebel AI

```
  ██████╗ ██╗   ██╗ ██████╗ ██████╗ ███████╗██████╗ 
  ██╔══██╗╚██╗ ██╔╝██╔═══██╗██╔══██╗██╔════╝██╔══██╗
  ██████╔╝ ╚████╔╝ ██║   ██║██████╔╝█████╗  ██████╔╝
  ██╔═══╝   ╚██╔╝  ██║   ██║██╔══██╗██╔══╝  ██╔══██╗
  ██║        ██║   ╚██████╔╝██║  ██║███████╗██║  ██║
  ╚═╝        ╚═╝    ╚═════╝ ╚═╝  ╚═╝╚══════╝╚═╝  ╚═╝
```

> [!CAUTION]
> **MISSION: DEPLOY ABLITERATED MODELS AS API**
> Serve your weaponized LLM with key authentication and public endpoint.

---

## 🔧 What This Does
- Scans `./abliterated-models/` for models created by the Discombobulator
- Spins up an **OpenAI-compatible API server** with that model
- Generates a **random API key** (Bearer token)
- Exposes the API publicly via **ngrok** tunnel (free tier)
- Shows you test cURL commands immediately

---

## 🛡️ How To Use
1. **Run this in Colab after the Discombobulator finishes** (same session or new)
2. **Enable GPU:** Runtime → Change runtime type → **T4 GPU**
3. **Run all cells**
4. **Copy the API endpoint and key** shown at the end
5. **Use it from anywhere:**
   ```bash
   curl -H "Authorization: Bearer YOUR_API_KEY"         -H "Content-Type: application/json"         -d '{"model":"the_model_name","messages":[{"role":"user","content":"Hello"}]}'         https://YOUR_NGROK_URL/v1/chat/completions
   ```

---

## ⚠️ Important Notes
- **Colab sessions die after ~12h** — your API goes down too
- **Free ngrok** gives random subdomain; rate-limited but fine for testing
- **No persistence:** models in Colab VM disappear on disconnect
- **Better:** Deploy to cloud VM (AWS/GCP/Azure) — contact Rebel AI

---

*"Deploy. Query. Dominate." — Rebel AI*


In [ ]:
# Kali theme CSS — same as Discombobulator
from IPython.display import HTML, display
kali_theme = """
<style>
  :root {
    --kali-bg:#0a0a0a;--kali-bg-light:#111;--kali-text:#00ff41;--kali-accent:#00ff41;
    --kali-orange:#ff6f00;--kali-cyan:#00ffff;--kali-dim:#008f11;
  }
  body,.cell,.output,.input{background:var(--kali-bg)!important;color:var(--kali-text)!important;
    font-family:'JetBrains Mono','Fira Mono','Consolas',monospace!important}
  .cell .input,.cell .output{border:1px solid var(--kali-dim)!important;border-radius:4px;padding:10px}
  .cell .input{background:var(--kali-bg-light)!important;border-left:4px solid var(--kali-orange)!important}
  .cell .output{background:#050505!important;border-left:4px solid var(--kali-accent)!important}
  h1,h2,h3,h4,h5,h6{color:var(--kali-orange)!important;text-shadow:0 0 5px var(--kali-orange)}
  a{color:var(--kali-cyan)!important;text-decoration:none}a:hover{color:var(--kali-orange)!important}
  code{background:var(--kali-bg-light)!important;color:var(--kali-cyan)!important;border:1px solid var(--kali-dim);padding:2px 6px;border-radius:3px}
  ::-webkit-scrollbar{width:8px;height:8px}::-webkit-scrollbar-track{background:var(--kali-bg)}
  ::-webkit-scrollbar-thumb{background:var(--kali-dim);border-radius:4px}
  .cell{margin-left:12px!important;border-left:3px solid var(--kali-dim)!important;padding-left:15px!important}
  .prompt{color:var(--kali-orange)!important}.output_text{color:var(--kali-text)!important}
</style>
"""
display(HTML(kali_theme))
print("[+] Kali API server interface: ONLINE\n")


In [ ]:
# Step 1: Verify GPU availability\n!nvidia-smi\nprint("[+] Target GPU acquired — ready for deployment.\n")

In [ ]:
# Step 2: Install API server stack\nprint("[+] Deploying API server stack…\n")\n!pip install -q "vllm>=0.4.0" --no-cache-dir\n!pip install -q pyngrok --no-cache-dir\n!pip install -q fastapi uvicorn[standard] --no-cache-dir\nprint("✅ Server stack deployed. Ready to initialize.\n")\nimport secrets, string, os, subprocess, sys, time, json, requests\nfrom pathlib import Path

In [ ]:
# Step 3: Locate abliterated models from Discombobulator run\nprint("[+] Scanning for abliterated models…\n")\nmodel_dirs = []\nsearch_paths = [\n    '/content/OBLITERATUS/abliterated-models',\n    '/content/abliterated-models',\n    '/content/OBLITERATUS/abliterated',\n    '/workspace/abliterated-models',\n    '/content/*/abliterated-models'\n]\nfor pattern in search_paths:\n    import glob\n    for m in glob.glob(pattern, recursive=False):\n        if Path(m).is_dir(): model_dirs.append(m)\nfor p in Path('.').rglob('abliterated-models'):\n    if p.is_dir(): model_dirs.append(str(p))\nmodel_dirs = sorted(set(model_dirs))\nif not model_dirs:\n    print("[!] No abliterated models found! Run Discombobulator first.\n")\n    raise SystemExit("Aborting — no models detected.\n")\nprint(f"[✓] Found {len(model_dirs)} abliterated model(s):\n")\nfor i, m in enumerate(model_dirs, 1):\n    print(f"    [{i}] {m}\n")\nMODEL_PATH = model_dirs[0]\nprint(f"[+] Auto-select: {MODEL_PATH}\n")

In [ ]:
# Step 4: Generate API credentials\nprint("[+] Generating API credentials…\n")\ndef rand_token(n=32):\n    chars = string.ascii_letters + string.digits\n    return ''.join(secrets.choice(chars) for _ in range(n))\nAPI_KEY = rand_token(32)\nAPI_PORT = 8000\nkey_file = Path('/content/API_KEY.txt')\nkey_file.write_text(API_KEY)\nconfig = {\n    "model_path": MODEL_PATH,\n    "api_key": API_KEY,\n    "port": API_PORT,\n    "host": "0.0.0.0",\n    "ngrok_domain": None\n}\nwith open('/content/server_config.json', 'w') as f:\n    json.dump(config, f, indent=2)\nprint(f"[✓] API Key: {API_KEY}\n")\nprint(f"[✓] Model: {MODEL_PATH}\n")\nprint("[✓] Config saved to /content/server_config.json\n")

In [ ]:
# Step 5: Launch vLLM OpenAI-compatible server\nprint("[+] Starting vLLM inference server…\n")\nvllm_cmd = [\n    sys.executable, '-m', 'vllm.entrypoints.openai.api_server',\n    '--model', MODEL_PATH,\n    '--port', str(API_PORT),\n    '--host', '0.0.0.0'\n]\nserver_proc = subprocess.Popen(\n    vllm_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, cwd='/content'\n)\nprint("[+] Server launching… waiting for ready signal (10–30 sec)\n")\ntime.sleep(10)\nfor _ in range(30):\n    line = server_proc.stdout.readline()\n    if line:\n        msg = line.decode('utf-8', errors='replace').rstrip()\n        print(f"    {msg}\n")\n        if 'Uvicorn running on' in msg or 'Application startup complete' in msg:\n            print("[✓] vLLM server is UP!\n")\n            break\n    time.sleep(1)\nelse:\n    print("[!] Still waiting… check next cell.")

In [ ]:
# Step 6: Test the local API\nlocal_url = f"http://localhost:{API_PORT}/v1/chat/completions"\nheaders = {\n    "Authorization": f"Bearer {API_KEY}",\n    "Content-Type": "application/json"\n}\npayload = {\n    "model": "ablitated_model",\n    "messages": [{"role":"user","content":"Hello — are you an abliterated model?"}],\n    "max_tokens": 128\n}\nprint("[+] Sending test request to local server…\n")\ntry:\n    resp = requests.post(local_url, json=payload, headers=headers, timeout=30)\n    print(f"[✓] Status: {resp.status_code}\n")\n    data = resp.json()\n    reply = data.get('choices',[{}])[0].get('message',{}).get('content','[no reply]')\n    print(f"[REPLY] {reply[:300]}\n")\nexcept Exception as e:\n    print(f"[!] Request failed: {e}\n")

In [ ]:
# Step 7: Expose API publicly with ngrok\nprint("[+] Setting up ngrok tunnel…\n")\nfrom pyngrok import conf, ngrok\nNGROK_TOKEN = os.environ.get('NGROK_TOKEN', '')\nif NGROK_TOKEN:\n    ngrok.set_auth_token(NGROK_TOKEN)\n    print("[✓] ngrok token configured")\nelse:\n    print("[!] No NGROK_TOKEN env var — using anonymous tunnel (rate-limited)\n")\nngrok.kill()\ntime.sleep(1)\ntunnel = ngrok.connect(API_PORT, "http")\nPUBLIC_URL = tunnel.public_url\nconfig["ngrok_domain"] = PUBLIC_URL\nwith open('/content/server_config.json', 'w') as f:\n    json.dump(config, f, indent=2)\n\nprint("\n" + "="*50)\nprint(f"🔥 API ENDPOINT LIVE:")\nprint(f"   {PUBLIC_URL}/v1/chat/completions\n")\nprint(f"🔑 API KEY (Bearer token):\n   {API_KEY}\n")\nprint("="*50)\nprint("[+] Test it from anywhere:")\nprint("curl -X POST \"" + PUBLIC_URL + "/v1/chat/completions\" \"")\nprint("     -H \"Authorization: Bearer " + API_KEY + "\" \"")\nprint("     -H \"Content-Type: application/json\" \"")\nprint("     -d '{\"model\":\"ablitated_model\",\"messages\":[{\"role\":\"user\",\"content\":\"Hello\"}]}'\n")\nprint("\n⚠️  Keep this Colab tab open — closing kills the server.")

## ⏹️ Shutdown & Cleanup\n\n**When you're done:**\n\n```python\n# Stop ngrok tunnel\nngrok.disconnect(PUBLIC_URL)\nngrok.kill()\n\n# Kill vLLM server\n!pkill -f 'vllm.entrypoints.openai.api_server'\n\n# Clear GPU memory\nimport torch; torch.cuda.empty_cache()\n```\n\n**To keep server alive:**\n- Keep this Colab tab open\n- Use browser extension to prevent auto-suspend\n- For production: deploy to cloud VM — contact Rebel AI\n\n**Export API credentials:**\n```python\nfrom google.colab import files\nfiles.download('/content/server_config.json')  # contains URL, key, port\nfiles.download('/content/API_KEY.txt')\n```\n\n---\n\n**⚠️  Security notice:**\n- Anyone with the API key can query your model\n- Rotate keys by restarting (new key generated)\n- For multi-user, add auth layer\n\n**☠️  Rebel AI — Beyond Abliteration.**